# Exploration des Données : Élections Présidentielles 2022 — Tour 1 (Bureaux de vote, France entière)

Ce notebook explore les résultats du **1er tour** des élections présidentielles 2022 au niveau **bureau de vote** pour la **France entière**, filtré sur le département du Rhône (69).  
Source : [data.gouv.fr — resultats-par-niveau-burvot-t1-france-entiere.xlsx](https://static.data.gouv.fr/resources/election-presidentielle-des-10-et-24-avril-2022-resultats-definitifs-du-1er-tour/20220414-152612/resultats-par-niveau-burvot-t1-france-entiere.xlsx)  
Identifiant ressource : `98eb9dab-f328-4dee-ac08-ac17211357a8`  
Format : **XLSX** (31.9 Mo) → converti en **CSV** (`;` comme séparateur)


In [5]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
import openpyxl

pd.set_option('display.max_rows', 500)


In [6]:


# ── 1. PATH CONFIGURATION ───────────────
BASE_DATA_DIR = "../../data"

RAW_DIR    = os.path.join(BASE_DATA_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DATA_DIR, "bronze")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_DIR, exist_ok=True)

XLSX_URL = (
    "https://static.data.gouv.fr/resources/"
    "election-presidentielle-des-10-et-24-avril-2022-resultats-definitifs-du-1er-tour/"
    "20220414-152612/resultats-par-niveau-burvot-t1-france-entiere.xlsx"
)

path_xlsx_raw     = os.path.join(RAW_DIR, "burvot_t1_france_entiere.xlsx")
path_rhone_bronze = os.path.join(BRONZE_DIR, "bronze_burvot_t1_rhone_69.csv")



In [7]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
# We keep the source file exactly as it is.
if not os.path.exists(path_xlsx_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(XLSX_URL, timeout=180)
    resp.raise_for_status()
    with open(path_xlsx_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_xlsx_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


✅ Raw file already exists in ../../data/raw


In [10]:

# ── 3. BRONZE LAYER: Ingestion & Metadata ───────────────────────────────────
# Bronze is for "raw-ish" data in a readable format with audit columns.
print("\n🛠 Processing RAW -> BRONZE...")
df_all = pd.read_excel(path_xlsx_raw, header=0, dtype=str, engine="openpyxl")

# Identify the department column
col_dept = next(c for c in df_all.columns if any(kw in c.lower() for kw in ["département", "dept"]))

# Filter for Rhône (69)
df_bronze = df_all[df_all[col_dept].astype(str).str.strip().str.lstrip("0") == "69"].copy()
print("____________________")
print(df_bronze.columns.to_list())

# --- RECONSTRUCT CANDIDATE COLUMNS (Structural Ingestion) ---
CAND_FIELDS = ["N°Panneau", "Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]
cols = list(df_bronze.columns)
start_cand = next((i for i, c in enumerate(cols) if "panneau" in c.lower()), None)

if start_cand is not None:
    new_cols = cols[:start_cand]
    # Calculate candidates based on remaining original columns
    n_cand = (len(cols) - start_cand) // len(CAND_FIELDS)
    for k in range(n_cand):
        suffix = f".{k}" if k > 0 else ""
        for field in CAND_FIELDS:
            new_cols.append(f"{field}{suffix}")
    df_bronze.columns = new_cols

# --- ADD BRONZE METADATA ---
# Essential for the Medallion pattern to track data lineage
df_bronze['extraction_source_url'] = XLSX_URL
df_bronze['ingestion_timestamp']   = datetime.now().isoformat()
df_bronze['source_file_name']     = os.path.basename(path_xlsx_raw)

# ── 4. SAVE TO BRONZE ───────────────────────────────────────────────────────
df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")

print(f"✅ BRONZE dataset created: {path_rhone_bronze}")
print(f"📊 Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")


🛠 Processing RAW -> BRONZE...
____________________
Index(['Code du département', 'Libellé du département',
       'Code de la circonscription', 'Libellé de la circonscription',
       'Code de la commune', 'Libellé de la commune', 'Code du b.vote',
       'Inscrits', 'Abstentions', '% Abs/Ins',
       ...
       'Unnamed: 95', 'Unnamed: 96', 'Unnamed: 97', 'Unnamed: 98',
       'Unnamed: 99', 'Unnamed: 100', 'Unnamed: 101', 'Unnamed: 102',
       'Unnamed: 103', 'Unnamed: 104'],
      dtype='str', length=105)
✅ BRONZE dataset created: ../../data/bronze/bronze_burvot_t1_rhone_69.csv
📊 Rows: 1317 | Columns: 108


In [9]:
df_bronze.dtypes

Code du département              str
Libellé du département           str
Code de la circonscription       str
Libellé de la circonscription    str
Code de la commune               str
Libellé de la commune            str
Code du b.vote                   str
Inscrits                         str
Abstentions                      str
% Abs/Ins                        str
Votants                          str
% Vot/Ins                        str
Blancs                           str
% Blancs/Ins                     str
% Blancs/Vot                     str
Nuls                             str
% Nuls/Ins                       str
% Nuls/Vot                       str
Exprimés                         str
% Exp/Ins                        str
% Exp/Vot                        str
N°Panneau                        str
Sexe                             str
Nom                              str
Prénom                           str
Voix                             str
% Voix/Ins                       str
%